# R12 · Block distance vs whole-matrix distance (supplementary Fig S9)

How much does leaving out the between-block entries change the distance? Breast, seed-0 layout: in each of the three
largest niches, 1,000 random distinct pairs of index tiles (one generator, `default_rng(0)`, drawn niche after niche),
and for each pair the whole-matrix log-Euclidean distance d_W = ‖log C_i − log C_j‖_F / √p and the block distance
d_B (Spindle-naive's scan, over the niche's blocks). Both from vectors already in the cache, on the same scale; a few
pairs are recomputed from the covariances as a check (`block_distance.py`).

* **Prerequisites:** `R0_index_cache` (staged covariances + whole-matrix vectors, seed-0 Spindle-naive index).
* **Writes:** `results/block_distance/pairs.csv` (one row per tile pair), `niches.csv` (tiles and blocks of the
  plotted niches).
* **Runtime / memory:** about a minute; < 4 GB (whole-matrix vectors memory-mapped).
  `sbatch experiments/common/run_nb.sbatch experiments/compute/R12_block_distance.ipynb`

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets.yaml").exists())
sys.path.insert(0, str(ROOT / "experiments" / "common"))
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr

import block_distance as bd
import data
import results_io as rio

TOPIC = "block_distance"
KEY = "breast_cancer"
FORCE = False
OUT = rio.topic(TOPIC)
print(data.stamp())

{'tag': 'method-v2', 'describe': 'method-v3-3-g01dbbe8', 'commit': '01dbbe8'}


## Pairs and distances

In [2]:
if FORCE or not (OUT / "pairs.csv").exists():
    pairs, niches = bd.pairs(KEY)
    bd.check(KEY, pairs)
    tag = data.stamp()["tag"]
    rio.write_csv(pairs.assign(key=KEY, tag=tag), TOPIC, "pairs.csv",
                  f"{KEY}, seed-0 layout, the {bd.N_NICHES} largest niches: {bd.N_PAIRS} random distinct pairs of index "
                  "tiles per niche (default_rng(0), niche after niche); whole-matrix log-Euclidean distance d_whole and "
                  "block distance d_block (both / sqrt(p), shrunk covariances); tiles in prepare() order")
    rio.write_csv(niches.assign(key=KEY, tag=tag), TOPIC, "niches.csv",
                  "the plotted niches, largest first: index tiles, blocks, genes p")
pairs = pd.read_csv(OUT / "pairs.csv")
niches = pd.read_csv(OUT / "niches.csv")
niches

niche 0: d_W 1.07393 vs cached 1.07393 | d_B 0.91849 vs cached 0.91849
niche 0: d_W 1.36316 vs cached 1.36316 | d_B 1.24093 vs cached 1.24093
niche 1: d_W 1.23417 vs cached 1.23417 | d_B 0.94457 vs cached 0.94457
niche 1: d_W 1.37397 vs cached 1.37397 | d_B 1.33489 vs cached 1.33489
niche 2: d_W 0.98530 vs cached 0.98530 | d_B 0.81293 vs cached 0.81293
niche 2: d_W 1.43124 vs cached 1.43124 | d_B 1.24642 vs cached 1.24642
wrote results/block_distance/pairs.csv: 3000 rows
wrote results/block_distance/niches.csv: 3 rows


,niche,tiles,blocks,p,key,tag
0,0,664,16,313,breast_cancer,method-v2
1,1,493,15,313,breast_cancer,method-v2
2,2,411,17,313,breast_cancer,method-v2


## Agreement per niche (shown, not saved)

In [3]:
def summary(g):
    gap = g.d_whole - g.d_block
    return pd.Series({"pairs": len(g),
                      "pearson_r": pearsonr(g.d_whole, g.d_block)[0],
                      "spearman_rho": spearmanr(g.d_whole, g.d_block)[0],
                      "median_gap": float(np.median(gap)),
                      "median_ratio_block_whole": float(np.median(g.d_block / g.d_whole)),
                      "frac_block_le_whole": float(np.mean(g.d_block <= g.d_whole))})

pairs.groupby("niche", sort=False).apply(summary).round(3)

,pairs,pearson_r,spearman_rho,median_gap,median_ratio_block_whole,frac_block_le_whole
niche,,,,,,
0,1000.0,0.969,0.950,0.255,0.783,0.997
1,1000.0,0.978,0.942,0.300,0.803,0.997
2,1000.0,0.985,0.955,0.195,0.842,0.980


In [4]:
rio.write_manifest(TOPIC, "R12_block_distance.ipynb", inputs=["results/_cache/<key>/whole.npy (R0)",
                                                              "results/_cache/<key>/covs.npy (R0)",
                                                              "results/_cache/<key>/seed0/naive.pkl (R0)"])

wrote results/block_distance/manifest.json: 2 files; peak memory 1.48 GB


{'topic': 'block_distance',
 'notebook': 'R12_block_distance.ipynb',
 'date': '2026-11-11 19:49',
 'tag': 'method-v2',
 'describe': 'method-v3-3-g01dbbe8',
 'commit': '01dbbe8',
 'inputs': ['results/_cache/<key>/whole.npy (R0)',
  'results/_cache/<key>/covs.npy (R0)',
  'results/_cache/<key>/seed0/naive.pkl (R0)'],
 'peak_rss_gb': 1.48,
 'notes': '',
 'files': {'niches.csv': {'description': 'the plotted niches, largest first: index tiles, blocks, genes p',
   'rows': 3,
   'bytes': 140},
  'pairs.csv': {'description': 'breast_cancer, seed-0 layout, the 3 largest niches: 1000 random distinct pairs of index tiles per niche (default_rng(0), niche after niche); whole-matrix log-Euclidean distance d_whole and block distance d_block (both / sqrt(p), shrunk covariances); tiles in prepare() order',
   'rows': 3000,
   'bytes': 217128}}}